# Week 2: Build a Chatbot with Memory

**Lab companion to [week_02.md](../agenda/week_02.md)**

In this lab, you will:
1. Understand conversation history
2. Build a chatbot class with memory
3. Implement different memory strategies
4. Handle context limits

In [ ]:
# Setup
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI()

print("✓ Ready!")

## 1. The Problem: AI Has No Memory

Each API call is independent. The AI doesn't remember previous messages:

In [ ]:
# First message
response1 = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "My name is Alice."}]
)
print("AI:", response1.choices[0].message.content)

# Second message - new conversation, no memory!
response2 = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "What is my name?"}]
)
print("AI:", response2.choices[0].message.content)

## 2. The Solution: Send Conversation History

We must send all previous messages with each request:

In [ ]:
# Manual conversation history
messages = []

# First turn
messages.append({"role": "user", "content": "My name is Alice."})
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages
)
ai_reply = response.choices[0].message.content
messages.append({"role": "assistant", "content": ai_reply})
print("AI:", ai_reply)

# Second turn - now with history!
messages.append({"role": "user", "content": "What is my name?"})
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages
)
print("AI:", response.choices[0].message.content)

In [ ]:
# Let's see what we're sending
print("Conversation history:")
for msg in messages:
    print(f"  [{msg['role']}]: {msg['content'][:50]}...")

## 3. Build a Chatbot Class

Let's encapsulate this in a reusable class:

In [ ]:
class Chatbot:
    """A simple chatbot with conversation memory."""
    
    def __init__(self, system_prompt: str = "You are a helpful assistant."):
        self.client = OpenAI()
        self.system_prompt = system_prompt
        self.history = []
    
    def chat(self, message: str) -> str:
        """Send a message and get a response."""
        # Add user message to history
        self.history.append({"role": "user", "content": message})
        
        # Build messages with system prompt + history
        messages = [
            {"role": "system", "content": self.system_prompt}
        ] + self.history
        
        # Get response
        response = self.client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages
        )
        
        # Add assistant response to history
        ai_reply = response.choices[0].message.content
        self.history.append({"role": "assistant", "content": ai_reply})
        
        return ai_reply
    
    def reset(self):
        """Clear conversation history."""
        self.history = []

# Create and test
bot = Chatbot()
print(bot.chat("My name is Bob."))
print(bot.chat("What is my name?"))

In [ ]:
# Test with a custom personality
pirate_bot = Chatbot(system_prompt="You are a friendly pirate. Use pirate speak.")

print(pirate_bot.chat("Hello!"))
print()
print(pirate_bot.chat("What treasure are you seeking?"))

## 4. Problem: Context Window Limits

Models have token limits. If conversation gets too long, we need strategies:

In [ ]:
import tiktoken

def count_tokens(messages: list, model: str = "gpt-4o-mini") -> int:
    """Count tokens in a message list."""
    encoding = tiktoken.encoding_for_model(model)
    total = 0
    for msg in messages:
        total += 4  # Message overhead
        total += len(encoding.encode(msg.get("content", "")))
    return total

# Check our chatbot's token usage
print(f"Current conversation uses {count_tokens(bot.history)} tokens")
print(f"GPT-4o-mini context limit: 128,000 tokens")

## 5. Memory Strategy: Sliding Window

Keep only the last N messages:

In [ ]:
class SlidingWindowChatbot:
    """Chatbot that keeps only the last N messages."""
    
    def __init__(self, system_prompt: str = "You are helpful.", max_messages: int = 10):
        self.client = OpenAI()
        self.system_prompt = system_prompt
        self.history = []
        self.max_messages = max_messages
    
    def chat(self, message: str) -> str:
        self.history.append({"role": "user", "content": message})
        
        # Trim to max_messages
        if len(self.history) > self.max_messages:
            self.history = self.history[-self.max_messages:]
        
        messages = [{"role": "system", "content": self.system_prompt}] + self.history
        
        response = self.client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages
        )
        
        ai_reply = response.choices[0].message.content
        self.history.append({"role": "assistant", "content": ai_reply})
        
        return ai_reply

# Test - only keeps last 4 messages
bot = SlidingWindowChatbot(max_messages=4)
print(bot.chat("My name is Charlie."))
print(bot.chat("I like pizza."))
print(bot.chat("My favorite color is blue."))
print(bot.chat("I work as a doctor."))
# This might forget the name!
print("---")
print(bot.chat("What do you remember about me?"))

## 6. Memory Strategy: Summarization

Summarize old messages instead of discarding:

In [ ]:
class SummarizingChatbot:
    """Chatbot that summarizes old messages."""
    
    def __init__(self, max_messages: int = 6):
        self.client = OpenAI()
        self.history = []
        self.summary = ""
        self.max_messages = max_messages
    
    def _summarize(self, messages: list) -> str:
        """Summarize a list of messages."""
        conversation = "\n".join([
            f"{m['role']}: {m['content']}" for m in messages
        ])
        
        response = self.client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{
                "role": "user",
                "content": f"Summarize this conversation briefly, keeping key facts:\n\n{conversation}"
            }]
        )
        return response.choices[0].message.content
    
    def chat(self, message: str) -> str:
        self.history.append({"role": "user", "content": message})
        
        # If too many messages, summarize the old ones
        if len(self.history) > self.max_messages:
            # Summarize first half
            to_summarize = self.history[:len(self.history)//2]
            new_summary = self._summarize(to_summarize)
            
            if self.summary:
                self.summary = f"{self.summary}\n\nMore recently: {new_summary}"
            else:
                self.summary = new_summary
            
            # Keep only recent messages
            self.history = self.history[len(self.history)//2:]
        
        # Build prompt with summary + recent history
        system = "You are a helpful assistant."
        if self.summary:
            system += f"\n\nPrevious conversation summary: {self.summary}"
        
        messages = [{"role": "system", "content": system}] + self.history
        
        response = self.client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages
        )
        
        ai_reply = response.choices[0].message.content
        self.history.append({"role": "assistant", "content": ai_reply})
        
        return ai_reply

# Test
bot = SummarizingChatbot(max_messages=4)
print(bot.chat("My name is Diana."))
print(bot.chat("I live in Paris."))
print(bot.chat("I work as a chef."))
print(bot.chat("My favorite dish is ratatouille."))
print("--- After summarization ---")
print("Summary:", bot.summary)
print()
print(bot.chat("What do you remember about me?"))

## 7. Interactive Chat Loop

Build a real interactive chatbot:

In [ ]:
def interactive_chat():
    """Run an interactive chat session."""
    bot = Chatbot(system_prompt="You are a friendly AI assistant named Aria.")
    
    print("Chat with Aria! (type 'quit' to exit)\n")
    
    while True:
        user_input = input("You: ")
        
        if user_input.lower() in ['quit', 'exit', 'q']:
            print("Goodbye!")
            break
        
        response = bot.chat(user_input)
        print(f"Aria: {response}\n")

# Uncomment to run interactive chat
# interactive_chat()

## 8. Exercises

Try these challenges:

In [ ]:
# Exercise 1: Create a study buddy chatbot
# It should help explain concepts and quiz you

class StudyBuddy(Chatbot):
    def __init__(self):
        super().__init__(system_prompt="""
You are a study buddy. Help the user learn by:
1. Explaining concepts clearly
2. Asking follow-up questions to test understanding
3. Providing examples
4. Being encouraging
""")

buddy = StudyBuddy()
print(buddy.chat("Explain what a variable is in programming."))

<details>
<summary>💡 <b>Hint</b> (click to expand)</summary>

A study buddy chatbot should:
1. Have a friendly, encouraging persona
2. Remember the topic being studied
3. Ask follow-up questions to test understanding

</details>

<details>
<summary>✅ <b>Solution</b> (click to expand)</summary>

```python
class StudyBuddy(Chatbot):
    def __init__(self):
        super().__init__(
            system_prompt="""You are a friendly study buddy. Help the user learn by:
1. Explaining concepts clearly
2. Asking follow-up questions to test understanding
3. Giving encouragement
4. Suggesting related topics to explore"""
        )

buddy = StudyBuddy()
print(buddy.chat("Help me understand recursion"))
```

</details>

<details>
<summary>💡 <b>Hint</b> (click to expand)</summary>

A study buddy chatbot should:
1. Have a friendly, encouraging persona
2. Remember the topic being studied
3. Ask follow-up questions to test understanding

</details>

<details>
<summary>✅ <b>Solution</b> (click to expand)</summary>

```python
class StudyBuddy(Chatbot):
    def __init__(self):
        super().__init__(
            system_prompt="""You are a friendly study buddy. Help the user learn by:
1. Explaining concepts clearly
2. Asking follow-up questions to test understanding
3. Giving encouragement
4. Suggesting related topics to explore"""
        )

buddy = StudyBuddy()
print(buddy.chat("Help me understand recursion"))
```

</details>

In [ ]:
# Exercise 2: Add message count tracking
# How many messages have been exchanged?

class TrackedChatbot(Chatbot):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.message_count = 0
        self.total_tokens = 0
    
    def chat(self, message: str) -> str:
        self.message_count += 1
        return super().chat(message)
    
    def stats(self) -> dict:
        return {
            "messages": self.message_count,
            "history_length": len(self.history)
        }

bot = TrackedChatbot()
bot.chat("Hello!")
bot.chat("How are you?")
print("Stats:", bot.stats())

<details>
<summary>💡 <b>Hint</b> (click to expand)</summary>

Track the count in the chatbot class:
1. Add a counter attribute in __init__
2. Increment it in the chat() method
3. Add a method to get stats

</details>

<details>
<summary>✅ <b>Solution</b> (click to expand)</summary>

```python
class ChatbotWithStats(Chatbot):
    def __init__(self, system_prompt="You are a helpful assistant."):
        super().__init__(system_prompt)
        self.message_count = 0
        self.total_tokens = 0
    
    def chat(self, message: str) -> str:
        self.message_count += 1
        response = super().chat(message)
        return response
    
    def get_stats(self):
        return {
            "messages": self.message_count,
            "history_length": len(self.history)
        }

bot = ChatbotWithStats()
bot.chat("Hello!")
bot.chat("How are you?")
print(bot.get_stats())  # {"messages": 2, ...}
```

</details>

<details>
<summary>💡 <b>Hint</b> (click to expand)</summary>

Track the count in the chatbot class:
1. Add a counter attribute in __init__
2. Increment it in the chat() method
3. Add a method to get stats

</details>

<details>
<summary>✅ <b>Solution</b> (click to expand)</summary>

```python
class ChatbotWithStats(Chatbot):
    def __init__(self, system_prompt="You are a helpful assistant."):
        super().__init__(system_prompt)
        self.message_count = 0
        self.total_tokens = 0
    
    def chat(self, message: str) -> str:
        self.message_count += 1
        response = super().chat(message)
        return response
    
    def get_stats(self):
        return {
            "messages": self.message_count,
            "history_length": len(self.history)
        }

bot = ChatbotWithStats()
bot.chat("Hello!")
bot.chat("How are you?")
print(bot.get_stats())  # {"messages": 2, ...}
```

</details>

## Summary

You learned:
- ✅ Why AI needs conversation history
- ✅ How to build a chatbot class
- ✅ Sliding window memory
- ✅ Summarization memory
- ✅ Token counting and limits

**Next:** [Week 3 - Better Prompts & Structured Output](week_03_structured_output.ipynb)